In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt  
from sklearn.decomposition import PCA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from yellowbrick.features import PCA as PcaViz

# abrimos el archivo usando una función específica de pandas
raw_dataset = pd.read_csv("CKD_NHANES_2021_2023.csv")

print(raw_dataset)

In [ ]:
raw_dataset.info()

In [ ]:
raw_dataset.head()

In [ ]:
raw_dataset.describe()

In [ ]:

# 1. Define tu lista personalizable de valores nulos (¡modifícala a tu gusto!)
valores_nulos = ["unknown", "Unknown", "UNKNOWN", "n/a", "NA", "?", "","7","9",]

# 2. Reemplazamos esos valores por np.nan en una copia temporal
# Usamos regex=False para evitar problemas con caracteres especiales
raw_dataset_temp = raw_dataset.replace(valores_nulos, np.nan)

# 3. Calculamos el porcentaje de nulos por columna (variable)
porcentaje_nulos = (raw_dataset_temp.isnull().mean() * 100).round(2)

# 4. Mostramos el resultado ordenado de mayor a menor porcentaje
print(porcentaje_nulos.sort_values(ascending=False))

In [ ]:

# Generar el boxplot agrupado para las variables seleccionadas

Cuantitativa_continua=["poverty_income_ratio","bmi","weight_kg","height_cm","bp_systolic","bp_diastolic","serum_creatinine","blood_urea_nitrogen","albumin_serum","phosphorus","bicarbonate","calcium","uric_acid","urine_creatinine","albumin_creatinine_ratio","urine_albumin","egfr"]

raw_dataset[Cuantitativa_continua].boxplot(figsize=(10, 6), rot=45)

plt.title('Boxplots de Variables Continuas')
plt.ylabel('Valores')
plt.grid(True)
plt.show()

In [ ]:


# 1-3. Mismo cálculo de tu celda
valores_nulos = ["unknown", "Unknown", "UNKNOWN", "n/a", "NA", "?", "", "7", "9"]
raw_dataset_temp = raw_dataset.replace(valores_nulos, np.nan)
porcentaje_nulos = (raw_dataset_temp.isnull().mean() * 100).round(2)

# 4. Ordenado de mayor a menor, solo las variables con algún nulo
nulos = porcentaje_nulos[porcentaje_nulos > 0].sort_values(ascending=False)

# Los 3 primeros de un color y el resto de otro
colores = ["#eb6834"] * 3 + ["#2a78d6"] * (len(nulos) - 3)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(nulos.index[::-1], nulos.values[::-1], color=colores[::-1], height=0.65)   # [::-1]: la mayor queda arriba
for y, v in enumerate(nulos.values[::-1]):
    ax.text(v + 1, y, f"{v:.1f} %".replace(".", ","), va="center", fontsize=8.5)

ax.set_xlim(0, 105)
ax.set_xlabel(f"% de las {len(raw_dataset):,} filas sin dato".replace(",", "."))
ax.set_title(f"Valores nulos por variable ({len(nulos)} de {raw_dataset.shape[1]} columnas)", loc="left")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
#limpieza del dataset
Clean = raw_dataset

Cualitativa_Ordinal= ["education_level","insulin_use","ckd_stage"]
Cualitativa_Nominal= ["gender","ethnicity","diabetes_diagnosed","diabetes_pills","ever_smoked","current_smoker","ckd_present"]
Cuantitativa_discreta=["participant_id","age"]
Cuantitativa_continua=["poverty_income_ratio","bmi","weight_kg","height_cm","bp_systolic","bp_diastolic","serum_creatinine","blood_urea_nitrogen","albumin_serum","phosphorus","bicarbonate","calcium","uric_acid","urine_creatinine","albumin_creatinine_ratio","urine_albumin","egfr"]

#Comienzo por las Cuantitativas_continuas Rellenando por la media los valores nulos
for col in Cuantitativa_continua:
    Clean[col] = raw_dataset[col].fillna(raw_dataset[col].mean())

#En el caso de insulin_use, Tomo el por defecto de que Si no tenes diabetes. no tomas pastillas/insulina

Clean.loc[Clean['diabetes_diagnosed'] == 2.0, 'insulin_use'] = 2.0
Clean.loc[Clean['diabetes_diagnosed'] == 2.0, 'diabetes_pills'] = 2.0

#En el caso de Current_smoker, tomo en cuenta la pregunta de si Fumo alguna vez 

Clean.loc[(Clean['ever_smoked'] == 2.0) & (Clean['current_smoker'].isna()), 'current_smoker'] = 3.0

#A partir de ahi, consideramos que toda fila de la tabla que contenga un Not a number, se puede borrar para acomodar el dataset

Clean = Clean.dropna()

#Ahora nos deshacemos de los outliers 
for col in Cuantitativa_continua:
    limite_inferior = Clean[col].quantile(0.01)
    limite_superior = Clean[col].quantile(0.99)
    # Aplicar el recorte en la columna
    Clean[col] = Clean[col].clip(lower=limite_inferior, upper=limite_superior)

for col in Cuantitativa_discreta:
    limite_inferior = Clean[col].quantile(0.01)
    limite_superior = Clean[col].quantile(0.99)
    # Aplicar el recorte en la columna
    Clean[col] = Clean[col].clip(lower=limite_inferior, upper=limite_superior)


valores_nulos = ["unknown", "Unknownplt.show()", "UNKNOWN", "n/a", "NA", "?", "","7","9"]


# 2. Reemplazamos esos valores por np.nan en una copia temporal
# Usamos regex=False para evitar problemas con caracteres especiales
raw_dataset_temp = Clean.replace(valores_nulos, np.nan)

# 3. Calculamos el porcentaje de nulos por columna (variaClean)
porcentaje_nulos = (raw_dataset_temp.isnull().mean() * 100).round(2)

# 4. Mostramos el resultado ordenado de mayor a menor porcentaje
print(porcentaje_nulos.sort_values(ascending=False))
# Muestra todos los valores únicos y su frecuencia (incluyendo NaN)
#print(Clean[columna].value_counts(dropna=False))

In [ ]:

# Muestra todos los valores únicos y su frecuencia (incluyendo NaN)
columna = 'ckd_stage'
print(raw_dataset[columna].value_counts(dropna=False))

In [ ]:


CleanNumeric = Clean.select_dtypes(include=[np.number])
# Generamos la matriz de correlacion, teniendo en cuenta unicamente valores numericos
correlation_matrix = CleanNumeric.corr()

%matplotlib inline
#imprimimos la matriz como un heatmap
plt.figure(figsize=(16,12))
sns.heatmap(correlation_matrix, vmin=-1.0, vmax=1.0, center=0.0, annot=True, cmap='coolwarm')
plt.title('Heatmap de la Matriz de Correlación', fontsize=18)
plt.show()

In [ ]:

#ANALISIS MULTIVARIADO GONZA HP3. 
#edad, sexo, BMI, presión arterial, diabetes, tabaquismo y nivel socioeconómico . 
Variables = ["age","gender","bmi","bp_systolic","bp_diastolic","diabetes_diagnosed","ever_smoked","poverty_income_ratio","ckd_present"]
X = Clean[Variables]
X['gender'] = Clean['gender'].map({'Male': 0.0, 'Female': 1.0}).astype(float)#Convierto un string a dicotomica float
correlation_matrix = X.corr()
ckd_present_correlations = correlation_matrix['ckd_present'].drop('ckd_present')

# Ordena las correlaciones de mayor a menor
sorted_correlations = ckd_present_correlations.sort_values(ascending=False)

# Crea un gráfico de barras para visualizar las correlaciones
plt.figure(figsize=(10, 6))
plt.grid(True)
sns.barplot(x=sorted_correlations.values, y=sorted_correlations.index)
plt.title('Correlaciones con Ckd presente')
plt.xlabel('Correlación')
plt.ylabel('Variable')
plt.xlim([-1, 1])
plt.show()
print(sorted_correlations)

In [ ]:
variable_objetivo = 'ckd_stage_float'
mapped_ckd = Clean.select_dtypes(include=[np.number])
mapped_ckd['ckd_stage_float'] = (Clean['ckd_stage']
    .map({
    "No CKD": -1.0,
    "Stage 1 (Kidney Damage)": 1.0,
    "Stage 2 (Mildly Decreased)": 2.0,
    "Stage 3a (Mild-Moderate)": 3.0,
    "Stage 3b (Moderate-Severe)": 4.0,
    "Stage 4 (Severely Decreased)": 5.0,
    "Stage 5 (Kidney Failure)": 6.0,
    "Unknown": 0.0,
}).astype(float))
# Seleccionamos únicamente la columna de la variable elegida y la ordenamos de mayor a menor

#Clean['ckd_stage'].head()
tabla_correlacion = mapped_ckd.corr()[[variable_objetivo]].sort_values(by=variable_objetivo, ascending=False)
#print(mapped_ckd['age'].value_counts(dropna=False))
display(tabla_correlacion)



In [ ]:


# 1. Definir un tamaño más chico para la figura (ancho, alto en pulgadas)
plt.figure(figsize=(6, 4))

# 2. Inicializar el visualizador (puedes nombrar las clases de tu variable dicotómica)
viz = PcaViz(
    scale=True,         
    proj_features=True,  
    projection=2,        
    classes=['No CKD', 'CKD']  # Etiquetas opcionales para la leyenda
)

# 3. Pasar X y la variable dicotómica 'y' para colorear los puntos automáticamente
viz.fit_transform(X, Clean['ckd_present'])

# 4. Asegurar el tamaño y mostrar el gráfico
viz.ax.figure.set_size_inches(6, 4)
viz.show()

In [ ]:


# Supongamos que X son tus datos ya escalados
# Instanciamos PCA (si no especificas n_components, calcula tantas como dimensiones tenga X)
pca = PCA()
pca.fit(X)

# 1. Varianza explicada por cada dimensión individualmente (en porcentaje)
varianza_individual = pca.explained_variance_ratio_ * 100

# 2. Varianza acumulada desde 1 hasta N dimensiones (en porcentaje)
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_) * 100

# Mostrar el aporte de cada dimensión y el acumulado
for i, (ind, acum) in enumerate(
    zip(varianza_individual, varianza_acumulada), 1
):
  print(
      f"Dimensión {i}: aporta un {ind:.2f}% | Acumulado: {acum:.2f}% de la"
      " información"
  )

In [ ]:

# 1. Preparación de datos (Ejemplo con fórmula de R style)
# Asumimos un DataFrame llamado df con las columnas:
# Ckd_present, edad, bmi, presion_arterial, diabetes, tabaquismo, ses (nivel socioeconómico), sexo


modelo_formula = (
    "ckd_present ~ age + bmi + bp_systolic + bp_diastolic + "
    "poverty_income_ratio + C(gender) + C(diabetes_diagnosed) + C(ever_smoked)"
)


# Ajustamos el modelo GLM con familia Binomial (Logit)
modelo = smf.logit(formula=modelo_formula, data=X).fit()

# 2. Ver el resumen estadístico completo (Coeficientes, Error Estándar, z, p-valores, IC)
print(modelo.summary())

# 3. Calcular los Odds Ratios (OR) y sus intervalos de confianza al 95%
params = modelo.params
conf = modelo.conf_int()
conf['OR'] = np.exp(params)
conf.columns = ['IC_2.5%', 'IC_97.5%', 'Odds_Ratio']

print("\n--- ODDS RATIOS Y INTERVALOS DE CONFIANZA ---")
print(conf[['Odds_Ratio', 'IC_2.5%', 'IC_97.5%']])

In [ ]:
or_table = np.exp(modelo.conf_int())
or_table.columns = ['IC_2.5%', 'IC_97.5%']
or_table['Odds_Ratio'] = np.exp(modelo.params)

print(or_table)

In [ ]:

# Generar el boxplot agrupado para las variables seleccionadas
Clean[Cuantitativa_continua].boxplot(figsize=(10, 6), rot=45)

plt.title('Boxplots de Variables Continuas')
plt.ylabel('Valores')
plt.grid(True)
plt.show()

In [ ]:

pca = PCA()
pca.fit(X)

# 1. Varianza explicada por cada dimensión individualmente (en porcentaje)
varianza_individual = pca.explained_variance_ratio_ * 100

# 2. Varianza acumulada desde 1 hasta N dimensiones (en porcentaje)
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_) * 100

# Mostrar el aporte de cada dimensión y el acumulado
for i, (ind, acum) in enumerate(
    zip(varianza_individual, varianza_acumulada), 1
):
  print(
      f"Dimensión {i}: aporta un {ind:.2f}% | Acumulado: {acum:.2f}% de la"
      " información"
  )

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from yellowbrick.features import PCA as PcaViz
features = [
    "bp_systolic",#
    "serum_creatinine",#
    "blood_urea_nitrogen",#
    "albumin_serum",#-
    "phosphorus",#-
    "bicarbonate",#
    "calcium",#-
    "uric_acid",#
    "albumin_creatinine_ratio",
    "egfr"
]

X = Clean[features]

mapping_grupos = { #no se notaba con tantos colores como categorias ya que se perdian las flechas
    "No CKD": "No CKD",
    "Stage 1 (Kidney Damage)": "Stage 1 al 3a",
    "Stage 2 (Mildly Decreased)": "Stage 1 al 3a",
    "Stage 3a (Mild-Moderate)": "Stage 1 al 3a",
    "Stage 3b (Moderate-Severe)": "Stage 3b al 5",
    "Stage 4 (Severely Decreased)": "Stage 3b al 5",
    "Stage 5 (Kidney Failure)": "Stage 3b al 5",
    "Unknown": "Unknown",
}

Clean["ckd_grouped"] = Clean["ckd_stage"].map(mapping_grupos)

y = (
    Clean["ckd_grouped"]
    .map({"No CKD": 0, "Stage 1 al 3a": 1, "Stage 3b al 5": 2, "Unknown": 3})
    .astype(int)
)
classes_names = ["No CKD", "Stage 1 al 3a", "Stage 3b al 5", "Unknown"]
plt.figure(figsize=(6, 4))

viz = PcaViz(
    scale=True,
    proj_features=True,
    projection=2,
    classes=classes_names,
)

viz.fit_transform(X, y)
viz.ax.figure.set_size_inches(6, 4)
viz.show()

In [ ]:
variable_objetivo = 'ckd_stage_float'
mapped_ckd = Clean[features]
ckd_positive = Clean.drop(Clean[Clean['ckd_stage'].isin(['No CKD', 'Unknown'])].index)
mapped_ckd['ckd_stage_float'] = (ckd_positive['ckd_stage']
    .map({
    "Stage 1 (Kidney Damage)": 1.0,
    "Stage 2 (Mildly Decreased)": 2.0,
    "Stage 3a (Mild-Moderate)": 3.0,
    "Stage 3b (Moderate-Severe)": 4.0,
    "Stage 4 (Severely Decreased)": 5.0,
    "Stage 5 (Kidney Failure)": 5.0,#Combino kidney failure con severly decrease por bajos datos
}).astype(float))
mapped_ckd = mapped_ckd.dropna()
# Seleccionamos únicamente la columna de la variable elegida y la ordenamos de mayor a menor

#Clean['ckd_stage'].head()
tabla_correlacion = mapped_ckd.corr()[[variable_objetivo]].sort_values(by=variable_objetivo, ascending=False)


tabla_correlacion = tabla_correlacion['ckd_stage_float'].drop('ckd_stage_float')

#print(mapped_ckd['age'].value_counts(dropna=False))
display(tabla_correlacion)

plt.figure(figsize=(10, 6))
plt.grid(True)
sns.barplot(x=tabla_correlacion.values, y=tabla_correlacion.index)
plt.title('Correlaciones con Ckd presente')
plt.xlabel('Correlación')
plt.ylabel('Variable')
plt.xlim([-1, 1])
#plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
# Separamos nuestra variable independiente
X = mapped_ckd['blood_urea_nitrogen']
# y nuestra variable objetivo (variable dependiente)
y = mapped_ckd['ckd_stage_float']

#plt.scatter(mapped_ckd['blood_urea_nitrogen'],mapped_ckd['ckd_stage_float'], alpha=0.5)
#plt.xlabel('Blood urea nitrogen')
#plt.ylabel('ckd_stage')
#plt.show()

sns.boxplot(x=mapped_ckd['ckd_stage_float'], y=mapped_ckd['blood_urea_nitrogen'])
plt.xlabel('ckd_stage')
plt.ylabel('Blood urea nitrogen')
plt.show()

In [ ]:


grupos = [g['blood_urea_nitrogen'].values
          for _, g in mapped_ckd.groupby('ckd_stage_float')]

H, p = stats.kruskal(*grupos)
print(f"Kruskal-Wallis: H = {H:.3f}, p = {p:.4g}")


In [ ]:
from statsmodels.miscmodels.ordinal_model import OrderedModel

# Sin constante: OrderedModel estima los umbrales por su cuenta
X = mapped_ckd[['blood_urea_nitrogen']]
y = mapped_ckd['ckd_stage_float'].astype(int)

modelo_ord = OrderedModel(y, X, distr='logit')
res_ord = modelo_ord.fit(method='bfgs', disp=False)
print(res_ord.summary())

# Odds ratio por cada unidad de BUN
print("Odds ratio:", np.exp(res_ord.params['blood_urea_nitrogen']))

In [ ]:
import numpy as np
import statsmodels.api as sm

# Separamos nuestra variable independiente
X = mapped_ckd['blood_urea_nitrogen'] #Cuantitativa continua
# y nuestra variable objetivo (variable dependiente)
y = mapped_ckd['ckd_stage_float'] #Cuantitativa_discreta

# Agregamos una constante como característica, para estimar la ordenada al origen
X = sm.add_constant(X)
# Ajustamos el modelo
model = sm.OLS(y, X).fit()

#print(mapped_ckd['ckd_stage_float'].value_counts(dropna=False))
# Imprimimos el resumen de los resultados
print(model.summary())
#print(y)

In [ ]:
model.resid

In [ ]:
sm.qqplot(model.resid, line='45')
plt.title("QQ-Plot de los residuos")
plt.show()

In [ ]:
from scipy.stats import shapiro

# aplicamos Shapiro-Wilk sobre los residuos
_, p = shapiro(model.resid)
print("p = %.4f" % (p))
# comparamos con el alpha
alpha = 0.05
if p > alpha:
    print("Se distribuyen normal (falló en rechazarse la hipótesis nula)")
else:
    print('No se distribuyen normal (se rechazó la hipótesis nula)')

In [ ]:


# 2. Usar el nombre de la columna o el índice de las columnas si son numéricos
counts = Clean['ckd_stage'].value_counts().reset_index()
counts.columns = ['ckd_stage', 'count']

counts.plot.bar(x='ckd_stage', y='count', color='skyblue')
plt.show()   

In [ ]:


# 1. Asegurar la ejecución del preprocesamiento y limpieza sobre el dataset
# (Asumiendo que 'Clean' ya fue procesado con la celda provista)
df_clean = Clean.copy()

# Si 'unknown' u otras variantes quedaron en ckd_stage o ethnicity, filtrar casos no válidos
df_clean = df_clean[~df_clean['ckd_stage'].astype(str).str.lower().str.startswith('unknown')]

# 2. Filtrar los casos identificados con CKD (ckd_present == 1)
ckd_cases = df_clean[df_clean['ckd_present'] == 1]

# 3. Distribución de frecuencias y proporciones por etnia en los casos identificados
frecuencias_etnia = ckd_cases['ethnicity'].value_counts()
porcentajes_etnia = ckd_cases['ethnicity'].value_counts(normalize=True) * 100

resumen_etnia = pd.DataFrame({
    'Casos': frecuencias_etnia,
    'Porcentaje (%)': porcentajes_etnia.round(2)
})
print("--- Distribución de casos de CKD por Etnia ---")
print(resumen_etnia)

# 4. Tabla de contingencia bivariada (ethnicity vs ckd_present)
tabla_contingencia = pd.crosstab(df_clean['ethnicity'], df_clean['ckd_present'], margins=True)
print("\n--- Tabla de Contingencia (ethnicity vs ckd_present) ---")
print(tabla_contingencia)

# 5. Test de Independencia Chi-cuadrado
obs = pd.crosstab(df_clean['ethnicity'], df_clean['ckd_present'])
chi2_stat, p_val, dof, expected = stats.chi2_contingency(obs)

alpha = 0.05
chi2_critico = stats.chi2.ppf(1 - alpha, dof)

print("\n--- Test Chi-cuadrado de Independencia ---")
print(f"Chi2 Observado: {chi2_stat:.4f}")
print(f"Grados de libertad: {dof}")
print(f"Valor crítico (alpha=0.05): {chi2_critico:.4f}")
print(f"p-valor: {p_val:.4e}")

In [ ]:


# df_clean proviene de la celda de limpieza
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x='ethnicity', hue='ckd_present', palette='Set2')
plt.title('Distribución de CKD por Etnia (Frecuencias Observadas)')
plt.xlabel('Etnia')
plt.ylabel('Cantidad de Individuos')
plt.xticks(rotation=25, ha='right')
plt.legend(title='CKD Present', labels=['No', 'Sí'])
plt.tight_layout()
plt.show()

In [ ]:
# 1. Asegurar la muestra limpia y válida
df_clean = Clean.copy()
df_clean = df_clean[~df_clean['ckd_stage'].astype(str).str.lower().str.startswith('unknown')]

# Separar los dos grupos por presencia/ausencia de daño renal
grupo_sano = df_clean[df_clean['ckd_present'] == 0]
grupo_ckd = df_clean[df_clean['ckd_present'] == 1]

variables_pa = ['bp_systolic', 'bp_diastolic']

print("=== 1. ESTADÍSTICAS DESCRIPTIVAS POR GRUPO ===")
for var in variables_pa:
    resumen = df_clean.groupby('ckd_present')[var].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    print(f"\nEstadísticos para {var}:")
    print(resumen.round(2))

# 2. VALIDACIÓN DE SUPUESTOS
print("\n=== 2. COMPROBACIÓN DE SUPUESTOS ===")
for var in variables_pa:
    print(f"\n--- Variable: {var} ---")
    
    # a) Normalidad: Kolmogorov-Smirnov / Anderson-Darling (n > 50)
    # Estandarizamos para contrastar contra la distribución normal teórica N(0,1)
    sano_std = (grupo_sano[var] - grupo_sano[var].mean()) / grupo_sano[var].std()
    ckd_std = (grupo_ckd[var] - grupo_ckd[var].mean()) / grupo_ckd[var].std()
    
    ks_sano = stats.kstest(sano_std, 'norm')
    ks_ckd = stats.kstest(ckd_std, 'norm')
    
    print(f"K-S Grupo Sano (No CKD): stat={ks_sano.statistic:.4f}, p-valor={ks_sano.pvalue:.4e}")
    print(f"K-S Grupo CKD: stat={ks_ckd.statistic:.4f}, p-valor={ks_ckd.pvalue:.4e}")
    
    # b) Homocedasticidad: Test de Levene
    levene_res = stats.levene(grupo_sano[var], grupo_ckd[var], center='median')
    print(f"Test de Levene (Homocedasticidad): stat={levene_res.statistic:.4f}, p-valor={levene_res.pvalue:.4e}")

# 3. CONTRASTES DE HIPÓTESIS BIVARIADOS
print("\n=== 3. PRUEBAS ESTADÍSTICAS BIVARIADAS ===")
for var in variables_pa:
    print(f"\n--- Resultados para {var} ---")
    
    # Test t con corrección de Welch (no asume varianzas iguales)
    t_stat, p_t = stats.ttest_ind(grupo_ckd[var], grupo_sano[var], equal_var=False, alternative='greater')
    print(f"Test t de Welch (una cola: CKD > No CKD): t={t_stat:.4f}, p-valor={p_t:.4e}")
    
    # Alternativa No Paramétrica: U de Mann-Whitney
    u_stat, p_u = stats.mannwhitneyu(grupo_ckd[var], grupo_sano[var], alternative='greater')
    print(f"Mann-Whitney U (una cola: CKD > No CKD): U={u_stat:.1f}, p-valor={p_u:.4e}")

In [ ]:


orden = ["Stage 1 (Kidney Damage)", 
         "Stage 2 (Mildly Decreased)", 
         "Stage 3a (Mild-Moderate)",
         "Stage 3b (Moderate-Severe)", 
         "Stage 4 (Severely Decreased)", 
         "Stage 5 (Kidney Failure)"]
etiq = ["Etapa 1", "Etapa 2", "Etapa 3a", "Etapa 3b", "Etapa 4", "Etapa 5"]

c = Clean[Clean.ckd_stage.isin(orden)]                 # excluye No CKD y Unknown
conteo = c.ckd_stage.value_counts().reindex(orden)
pct = conteo / conteo.sum() * 100

fig, ax = plt.subplots(figsize=(9, 5))
barras = ax.bar(etiq, pct, color="#2a78d6", width=0.6)
for r, v in zip(barras, pct):
    ax.text(r.get_x() + r.get_width() / 2, v + 1, f"{v:.1f} %", ha="center", fontsize=9)

ax.axhline(50, color="gray", ls="--", lw=1)            # línea de la mayoría
ax.set_ylim(0, 80)
ax.set_ylabel("% de los casos con CKD")
ax.grid(axis="y", alpha=0.3)
ax.set_title(f"Distribución porcentual de las etapas de CKD (n = {conteo.sum()})")
plt.tight_layout()
plt.show()

In [ ]:


var1, var2 = "poverty_income_ratio", "ckd_present"
d = Clean.dropna(subset=[var1, var2])

con = d.loc[d[var2] == 1, var1]
sin = d.loc[d[var2] == 0, var1]

u, p = stats.mannwhitneyu(con, sin, alternative="two-sided")
print(f"n con CKD = {len(con)}, n sin CKD = {len(sin)}")
print(f"Mediana con CKD = {con.median():.2f}, sin CKD = {sin.median():.2f}")
print(f"U = {u:.0f}, p = {p:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot para Sistólica (sin warning)
sns.boxplot(
    ax=axes[0], 

    data=df_clean, 
    x='ckd_present', 
    y='bp_systolic', 
    hue='ckd_present', 
    palette='Set2', 
    legend=False
)
axes[0].set_title('Presión Sistólica vs Presencia de CKD')
axes[0].set_xlabel('CKD Presente (0 = No, 1 = Sí)')
axes[0].set_ylabel('Presión Sistólica (mmHg)')

# Boxplot para Diastólica (sin warning)
sns.boxplot(
    ax=axes[1], 
    data=df_clean, 
    x='ckd_present', 
    y='bp_diastolic', 
    hue='ckd_present', 
    palette='Set2', 
    legend=False
)
axes[1].set_title('Presión Diastólica vs Presencia de CKD')
axes[1].set_xlabel('CKD Presente (0 = No, 1 = Sí)')
axes[1].set_ylabel('Presión Diastólica (mmHg)')

plt.tight_layout()
plt.show()

In [1]:
# 1. Asegurar la cohorte válida de análisis sobre df_clean
df_analisis = Clean[
    ~Clean['ckd_stage'].astype(str).str.lower().str.startswith('unknown')
].copy()

# Crear etiquetas descriptivas para los ejes
df_analisis['Condicion_CKD'] = df_analisis['ckd_present'].map(
    {0: 'Sin CKD', 1: 'Con CKD'}
)

# 2. Configurar figura con 2 paneles complementarios
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- PANEL 1: Boxplot Comparativo con Medias (directo para ANOVA) ---
sns.boxplot(
    data=df_analisis,
    x='Condicion_CKD',
    y='poverty_income_ratio',
    hue='Condicion_CKD',
    palette=['#5dade2', '#ec7063'],
    legend=False,
    showmeans=True,  # Grafica la media como marcador (clave para ANOVA)
    meanprops={
        'marker': 'D',
        'markeredgecolor': 'black',
        'markerfacecolor': 'yellow',
        'markersize': 7,
    },
    ax=axes[0],
)

axes[0].set_title(
    'Distribución de Ratio de Pobreza según Condición de CKD',
    fontsize=12,
    fontweight='bold',
)
axes[0].set_xlabel('Diagnóstico Renal', fontsize=11)
axes[0].set_ylabel(
    'Ratio Pobreza / Ingresos (poverty_income_ratio)', fontsize=11
)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# --- PANEL 2: Curvas de Densidad (KDE) solapadas ---
sns.kdeplot(
    data=df_analisis,
    x='poverty_income_ratio',
    hue='Condicion_CKD',
    palette=['#2980b9', '#c0392b'],
    common_norm=False,
    fill=True,
    alpha=0.35,
    ax=axes[1],
)

axes[1].set_title(
    'Densidad de Probabilidad del Ratio de Pobreza',
    fontsize=12,
    fontweight='bold',
)
axes[1].set_xlabel(
    'Ratio Pobreza / Ingresos (poverty_income_ratio)', fontsize=11
)
axes[1].set_ylabel('Densidad', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

NameError: name 'Clean' is not defined

In [ ]:
# Filtramos la cohorte válida de análisis
df_eda = Clean[
    ~Clean['ckd_stage'].astype(str).str.lower().str.startswith('unknown')
].copy()
df_eda['Estado'] = df_eda['ckd_present'].map({0: 'Sano', 1: 'Con CKD'})

# FIGURA 1: Etnia vs Diagnóstico
plt.figure(figsize=(9, 4.5))
sns.countplot(
    data=df_eda,
    x='ethnicity',
    hue='Estado',
    palette=['#5dade2', '#ec7063'],
    legend=True,
)
plt.title('Participantes según Etnia y Presencia de Daño Renal', fontsize=12)
plt.xlabel('Etnia', fontsize=10)
plt.ylabel('Cantidad de Personas', fontsize=10)
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# FIGURA 2: Boxplot de Presión Sistólica
plt.figure(figsize=(6, 4.5))
sns.boxplot(
    data=df_eda,
    x='Estado',
    y='bp_systolic',
    palette=['#5dade2', '#ec7063'],
    showmeans=True,
    meanprops={
        'marker': 'o',
        'markeredgecolor': 'black',
        'markerfacecolor': 'yellow',
    },
)
plt.title('Niveles de Presión Sistólica según Diagnóstico', fontsize=12)
plt.xlabel('Condición Renal', fontsize=10)
plt.ylabel('Presión Sistólica (mmHg)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# FIGURA 3: Distribución por Estadios Clínicos (Solo casos positivos)
df_casos = df_eda[df_eda['ckd_present'] == 1]
orden_estadios = [
    'Stage 1 (Kidney Damage)',
    'Stage 2 (Mildly Decreased)',
    'Stage 3a (Mild-Moderate)',
    'Stage 3b (Moderate-Severe)',
    'Stage 4 (Severely Decreased)',
    'Stage 5 (Kidney Failure)',
]
conteo_estadios = (
    df_casos['ckd_stage'].value_counts(normalize=True).loc[orden_estadios] * 100
)

plt.figure(figsize=(9, 4.5))
sns.barplot(
    x=conteo_estadios.index,
    y=conteo_estadios.values,
    color='#3498db',
)
plt.title('Distribución Porcentual por Estadio de CKD', fontsize=12)
plt.xlabel('Estadio Clínico', fontsize=10)
plt.ylabel('Porcentaje del Total (%)', fontsize=10)
plt.xticks(rotation=25, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()